## This notebook works best with GPU

GPU (T4) significantly speeds up inference. CPU works but will be slower.

**Runtime > Change Runtime Type > T4 GPU**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msds-marketing-analytics/colab-notebooks/blob/main/LLMs/MSDSTextClassification_LLMInferencing.ipynb)

# LLM Text Classification: Zero-Shot, Few-Shot, and Prompt Engineering

Traditional text classification requires training a model on labeled data.
Modern LLMs offer an alternative: classify text through **prompting** — no
task-specific training required. This notebook explores that paradigm.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Use zero-shot classification** with an NLI model to classify text without any examples
2. **Design prompts** for generative LLMs to perform classification tasks
3. **Apply few-shot prompting** to improve classification by providing examples in context
4. **Engineer prompts** systematically and measure how format affects accuracy
5. **Compare prompting approaches** with fine-tuned models on the same data

## Approach

We evaluate every approach on the same 200 IMDB reviews so results are
directly comparable. Three models are used:

- **DistilBART-MNLI**: A distilled NLI model for zero-shot classification
- **Flan-T5-base**: A small instruction-tuned generative model for prompt-based classification
- **DistilBERT-SST2**: A fine-tuned sentiment classifier as a reference baseline

In [ ]:
!pip install -q datasets transformers torch matplotlib seaborn scikit-learn

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

# Seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

---
## Part 1: Data Setup

We use 200 IMDB movie reviews as our test set throughout the notebook.
Every approach is evaluated on the same examples so results are directly
comparable.

In [ ]:
# Load test data (same 200 examples used throughout)
dataset = load_dataset("imdb", split="test").shuffle(seed=42).select(range(200))
texts = list(dataset["text"])
true_labels = np.array(dataset["label"])  # 0 = negative, 1 = positive

print(f"Test set: {len(texts)} examples")
print(f"  Negative: {(true_labels == 0).sum()}")
print(f"  Positive: {(true_labels == 1).sum()}")
print(f"\nSample review (first 200 chars):")
print(f"  [{['negative', 'positive'][true_labels[0]]}] '{texts[0][:200]}...'")

---
## Part 2: Zero-Shot Classification with NLI

### How It Works

Zero-shot classification repurposes a **Natural Language Inference (NLI)** model.
NLI models are trained to determine whether a hypothesis follows from a premise.
For classification, we frame each label as a hypothesis:

- Premise: *"This movie was terrible and boring."*
- Hypothesis 1: *"This text is about something positive."* → **contradiction**
- Hypothesis 2: *"This text is about something negative."* → **entailment**

The model picks the label whose hypothesis has the highest entailment score.
No examples needed — just describe the labels.

We use `valhalla/distilbart-mnli-12-3`, a distilled version of BART trained on
the Multi-Genre NLI dataset. It is much faster than the full-size BART-MNLI while
retaining most of its accuracy.

In [ ]:
# Load a distilled BART-MNLI model (much faster than bart-large-mnli)
nli_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3",
    device=device
)

# Demonstrate on a single example
demo_text = texts[0][:300]
demo_result = nli_classifier(demo_text, ["positive", "negative"])
print(f"Text: '{demo_text[:100]}...'")
print(f"Prediction: {demo_result['labels'][0]} (score: {demo_result['scores'][0]:.3f})")
print(f"All scores: {dict(zip(demo_result['labels'], [f'{s:.3f}' for s in demo_result['scores']]))}")

In [ ]:
# Run zero-shot on all 200 examples
print("Running zero-shot classification on 200 examples...")
start = time.time()

nli_results = nli_classifier(
    texts, ["positive", "negative"],
    batch_size=16, truncation=True, max_length=512
)

nli_time = time.time() - start
nli_preds = np.array([1 if r["labels"][0] == "positive" else 0 for r in nli_results])

print(f"Done in {nli_time:.1f}s")
print(f"\nZero-Shot Classification (BART-MNLI)")
print("=" * 55)
print(classification_report(true_labels, nli_preds, target_names=["Negative", "Positive"]))
nli_acc = accuracy_score(true_labels, nli_preds)
print(f"Accuracy: {nli_acc:.3f}")

---
## Part 3: Prompt-Based Classification with a Generative LLM

Instead of an NLI model, we can use a **generative LLM** and ask it to
classify text by constructing a prompt. The model generates a response
("positive" or "negative") that we parse as the prediction.

We use `google/flan-t5-base`, a 250M-parameter encoder-decoder model that was
instruction-tuned on many tasks including classification. It is small enough to
run quickly but capable enough to follow prompts.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load Flan-T5-base directly (encoder-decoder models need Seq2Seq, not the
# text-generation pipeline which only supports decoder-only models)
t5_name = "google/flan-t5-base"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name)
t5_device = "cuda" if torch.cuda.is_available() else "cpu"
t5_model.to(t5_device)
t5_model.eval()

print(f"Loaded {t5_name} ({sum(p.numel() for p in t5_model.parameters()):,} parameters)")

def generate(prompt, max_new_tokens=5):
    """Generate a response from Flan-T5."""
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(t5_device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = t5_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)

def classify_with_generator(texts, build_prompt):
    """Run prompt-based classification on a list of texts.

    build_prompt: function that takes text and returns a prompt string.
    Returns predictions (1=positive, 0=negative, -1=unparseable) and raw outputs.
    """
    predictions = []
    raw_outputs = []
    for text in texts:
        prompt = build_prompt(text)
        output = generate(prompt).strip().lower()
        raw_outputs.append(output)
        if "positive" in output:
            predictions.append(1)
        elif "negative" in output:
            predictions.append(0)
        else:
            predictions.append(-1)
    return np.array(predictions), raw_outputs

In [ ]:
# Define our basic prompt template
def basic_prompt(text):
    return (
        "Classify the sentiment of this movie review as positive or negative.\n"
        f"Review: {text[:500]}\n"
        "Sentiment:"
    )

# Demonstrate on a single example so we can see what the model outputs
sample_prompt = basic_prompt(texts[0])
print("Sample prompt (truncated):")
print("-" * 40)
print(sample_prompt[:250] + "...")
print("-" * 40)

sample_output = generate(sample_prompt)
print(f"\nModel output: '{sample_output}'")

In [ ]:
# Run prompt-based classification on all 200 examples
print("Running prompt-based classification on 200 examples...")
start = time.time()

prompt_preds, prompt_outputs = classify_with_generator(texts, basic_prompt)

prompt_time = time.time() - start

# Report unparseable outputs
n_unparseable = (prompt_preds == -1).sum()
if n_unparseable > 0:
    print(f"\nWarning: {n_unparseable} outputs could not be parsed as positive/negative")
    unparseable_examples = [o for p, o in zip(prompt_preds, prompt_outputs) if p == -1]
    for o in unparseable_examples[:5]:
        print(f"  Raw output: '{o}'")

# Evaluate on parseable outputs only
parseable = prompt_preds != -1
print(f"\nDone in {prompt_time:.1f}s ({parseable.sum()}/{len(texts)} parseable)")
print(f"\nPrompt-Based Classification (Flan-T5-base)")
print("=" * 55)
print(classification_report(
    true_labels[parseable], prompt_preds[parseable],
    target_names=["Negative", "Positive"]
))
prompt_acc = accuracy_score(true_labels[parseable], prompt_preds[parseable])
print(f"Accuracy: {prompt_acc:.3f} ({parseable.sum()} parseable out of {len(texts)})")

---
## Part 4: Few-Shot Prompting

Zero-shot prompting gives the model no examples of the task. **Few-shot
prompting** includes a small number of labeled examples in the prompt itself,
showing the model exactly what format to follow.

Few-shot prompting often improves accuracy, but not always. Models that were
already instruction-tuned on classification tasks (like Flan-T5) may already
"know" the task format, so the examples add less. The benefit tends to be
larger for tasks the model hasn't seen during training, or for more complex
label sets.

In [ ]:
# Load balanced few-shot examples from the training set
train_sample = load_dataset("imdb", split="train").shuffle(seed=42).select(range(50))
pos_ex = next(ex for ex in train_sample if ex["label"] == 1)
neg_ex = next(ex for ex in train_sample if ex["label"] == 0)

few_shot_demos = [
    (neg_ex["text"][:200], "negative"),
    (pos_ex["text"][:200], "positive"),
]

print("Few-shot examples:")
for demo_text, demo_label in few_shot_demos:
    print(f"  [{demo_label}] '{demo_text[:70]}...'")

# Build few-shot prompts with these 2 demonstration examples
def few_shot_prompt(text):
    prompt = "Classify the sentiment of movie reviews as positive or negative.\n\n"
    for demo_text, demo_label in few_shot_demos:
        prompt += f'Review: "{demo_text}"\nSentiment: {demo_label}\n\n'
    prompt += f'Review: "{text[:300]}"\nSentiment:'
    return prompt

# Show what a few-shot prompt looks like
sample_fs = few_shot_prompt(texts[0])
print(f"\nSample few-shot prompt (first 500 chars):")
print("-" * 40)
print(sample_fs[:500] + "...")
print("-" * 40)

In [ ]:
# Run few-shot classification
print("Running few-shot classification on 200 examples...")
start = time.time()

fs_preds, fs_outputs = classify_with_generator(texts, few_shot_prompt)

fs_time = time.time() - start
fs_parseable = fs_preds != -1

n_unparseable = (fs_preds == -1).sum()
if n_unparseable > 0:
    print(f"Warning: {n_unparseable} unparseable outputs")
    for o in [o for p, o in zip(fs_preds, fs_outputs) if p == -1][:3]:
        print(f"  Raw output: '{o}'")

print(f"\nDone in {fs_time:.1f}s ({fs_parseable.sum()}/{len(texts)} parseable)")
print(f"\nFew-Shot Classification (Flan-T5-base, 2 examples)")
print("=" * 55)
print(classification_report(
    true_labels[fs_parseable], fs_preds[fs_parseable],
    target_names=["Negative", "Positive"]
))
fs_acc = accuracy_score(true_labels[fs_parseable], fs_preds[fs_parseable])
print(f"Accuracy: {fs_acc:.3f}")

# Compare with zero-shot prompt
print(f"\nComparison:")
print(f"  Zero-shot prompt: {prompt_acc:.3f}")
print(f"  Few-shot (2 examples): {fs_acc:.3f}")
print(f"  Improvement: {fs_acc - prompt_acc:+.3f}")

---
## Part 5: Prompt Engineering

The way you phrase a prompt can significantly affect classification accuracy —
sometimes more than adding few-shot examples does. This section compares three
prompt formats on the same data. Notice that the simplest prompt may not be
the worst, and the most elaborate may not be the best.

In [ ]:
# Define alternative prompt formats
def minimal_prompt(text):
    return f"Positive or negative? {text[:500]}"

def structured_prompt(text):
    return (
        "Task: Sentiment Classification\n"
        "Options: positive, negative\n"
        f"Text: {text[:500]}\n"
        "Classification:"
    )

# We already have results for basic_prompt from Part 3.
# Run the other two formats.
prompt_formats = {
    "basic": {"fn": basic_prompt, "preds": prompt_preds, "time": prompt_time},
}

for name, fn in [("minimal", minimal_prompt), ("structured", structured_prompt)]:
    print(f"Running '{name}' prompt format...")
    start = time.time()
    preds, _ = classify_with_generator(texts, fn)
    elapsed = time.time() - start
    prompt_formats[name] = {"fn": fn, "preds": preds, "time": elapsed}
    print(f"  Done in {elapsed:.1f}s")

print("\nAll prompt formats complete.")

In [ ]:
# Compare prompt formats
print("Prompt Format Comparison")
print("=" * 55)
print(f"{'Format':<15} {'Accuracy':>10} {'Parseable':>10} {'Time (s)':>10}")
print("-" * 55)

format_accuracies = {}
for name, data in prompt_formats.items():
    p = data["preds"]
    mask = p != -1
    acc = accuracy_score(true_labels[mask], p[mask]) if mask.sum() > 0 else 0
    format_accuracies[name] = acc
    print(f"{name:<15} {acc:>10.3f} {mask.sum():>8}/{len(texts)} {data['time']:>10.1f}")

# Show what each format looks like for comparison
sample = texts[0][:100] + "..."
print(f"\nExample prompts for: '{sample[:60]}...'")
print()
for name, data in prompt_formats.items():
    prompt = data["fn"](texts[0])
    # Show first 120 chars of each prompt
    print(f"  [{name}]: {prompt[:120]}...")
    print()

---
## Part 6: Comparison with a Fine-Tuned Model

How do prompting approaches compare to a model that was actually **trained**
on sentiment data? We load `distilbert-base-uncased-finetuned-sst-2-english`,
a DistilBERT model fine-tuned on the Stanford Sentiment Treebank, and run it
on the same 200 examples.

This is not a fair fight — the fine-tuned model was trained specifically for
this task. But the results may surprise you: for straightforward binary
sentiment, the gap between prompting and fine-tuning can be smaller than
expected. The tradeoff is not just accuracy — it is also about flexibility,
cost of getting started, and how quickly you can adapt to new tasks.

In [ ]:
# Load fine-tuned sentiment classifier as reference
ft_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

print("Running fine-tuned DistilBERT on 200 examples...")
start = time.time()

ft_results = ft_classifier(texts, batch_size=32, truncation=True, max_length=512)

ft_time = time.time() - start
ft_preds = np.array([1 if r["label"] == "POSITIVE" else 0 for r in ft_results])
ft_acc = accuracy_score(true_labels, ft_preds)

print(f"Done in {ft_time:.1f}s")
print(f"\nFine-Tuned Classification (DistilBERT-SST2)")
print("=" * 55)
print(classification_report(true_labels, ft_preds, target_names=["Negative", "Positive"]))
print(f"Accuracy: {ft_acc:.3f}")

In [ ]:
# Head-to-head comparison of all approaches
best_prompt_name = max(format_accuracies, key=format_accuracies.get)

comparison = {
    "Zero-Shot (BART-MNLI)": {"acc": nli_acc, "time": nli_time, "training": "None"},
    f"Prompt (Flan-T5, {best_prompt_name})": {
        "acc": format_accuracies[best_prompt_name],
        "time": prompt_formats[best_prompt_name]["time"],
        "training": "None"
    },
    "Few-Shot (Flan-T5, 2 examples)": {"acc": fs_acc, "time": fs_time, "training": "None"},
    "Fine-Tuned (DistilBERT-SST2)": {"acc": ft_acc, "time": ft_time, "training": "67M params on SST-2"},
}

print("Head-to-Head Comparison")
print("=" * 70)
print(f"{'Approach':<35} {'Accuracy':>10} {'Time (s)':>10} {'Training Required':>15}")
print("-" * 70)
for name, data in comparison.items():
    print(f"{name:<35} {data['acc']:>10.3f} {data['time']:>10.1f} {data['training']:>15}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(comparison.keys())
accs = [d["acc"] for d in comparison.values()]
times = [d["time"] for d in comparison.values()]
colors = ["steelblue", "steelblue", "steelblue", "salmon"]

# Accuracy
axes[0].barh(range(len(names)), accs, color=colors, alpha=0.7)
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=8)
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Classification Accuracy")
axes[0].set_xlim(0, 1)
for i, acc in enumerate(accs):
    axes[0].text(acc + 0.01, i, f"{acc:.3f}", va="center", fontsize=9)

# Inference time
axes[1].barh(range(len(names)), times, color=colors, alpha=0.7)
axes[1].set_yticks(range(len(names)))
axes[1].set_yticklabels(names, fontsize=8)
axes[1].set_xlabel("Time (seconds)")
axes[1].set_title("Inference Time (200 examples)")
for i, t in enumerate(times):
    axes[1].text(t + 0.5, i, f"{t:.1f}s", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\nBlue = prompting approaches (no task-specific training)")
print("Red = fine-tuned model (trained on labeled sentiment data)")

---
## When to Use Each Approach

| Approach | Best For | Limitations |
|----------|----------|-------------|
| **Zero-shot (NLI)** | Quick prototyping, new tasks with no labeled data | Accuracy limited by label phrasing; no examples to learn from |
| **Prompt-based** | Tasks where you can describe the format; flexible label sets | Sensitive to prompt wording; limited by model's instruction-following |
| **Few-shot** | When you have a handful of examples but not enough to train | Prompt length limits how many examples fit; example selection matters |
| **Fine-tuned** | Production systems where accuracy is critical | Requires labeled data and training infrastructure; less flexible |

### Key Tradeoffs

- **Accuracy vs. flexibility**: Fine-tuned models win on accuracy but only work for
  the task they were trained on. Prompting approaches work on any task you can describe.

- **Cost of getting started**: Prompting approaches require zero training data and zero
  compute for training. Fine-tuning requires both.

- **Prompt sensitivity**: Small changes in prompt wording can cause large accuracy swings.
  The prompt engineering section showed this. Fine-tuned models have no prompt to worry about.

- **Inference cost**: Generative models are slower per example than fine-tuned classifiers
  because they generate tokens one at a time. NLI-based zero-shot is faster but still
  slower than a dedicated classifier.

- **Input length**: Prompting approaches consume part of the model's context window
  with instructions and examples, leaving less room for the actual text. Fine-tuned
  models use the full context for the input.

---
## Key Takeaways

1. **LLMs can classify text without task-specific training** — zero-shot and
   few-shot prompting enable classification on any task you can describe.

2. **NLI-based zero-shot classification** repurposes natural language inference
   models. You provide candidate labels; the model picks the most consistent one.

3. **Few-shot examples do not always help.** Instruction-tuned models like
   Flan-T5 may already understand the task format, so adding examples provides
   little benefit. The gain is typically larger for unfamiliar tasks or complex
   label sets.

4. **Prompt format matters — and simpler can win.** Different wordings of the
   same task can produce substantially different accuracy, and the most minimal
   prompt sometimes outperforms more elaborate ones. Prompt engineering is
   empirical, not intuitive.

5. **Fine-tuned models still win on accuracy** for well-defined tasks with
   available training data, but the margin may be narrower than expected for
   simple tasks like binary sentiment. The advantage of prompting is flexibility
   and zero training cost.

## Exercises

1. **More few-shot examples**: Try 4 or 6 examples instead of 2. Does accuracy
   keep improving? At what point does prompt length become a problem?

2. **Label phrasing**: In the NLI zero-shot classifier, try different label
   phrasings: "good" vs. "positive" vs. "favorable". How much does it matter?

3. **Different generative model**: Replace Flan-T5-base with Flan-T5-large
   or Flan-T5-small. How does model size affect prompt-based classification?

4. **Multi-class classification**: Try classifying AG News articles into
   4 categories (World, Sports, Business, Technology) using these same
   techniques. How do the approaches compare on a harder task?

5. **Example selection**: Instead of random few-shot examples, try selecting
   examples that are similar to the test input (e.g., by length or topic).
   Does targeted example selection improve accuracy?